# Advanced Python for Data Science Interviews
## Deep-Dive Companion Notebook

This notebook covers the Python concepts that **separate candidates who get offers from those who don't**.
Every topic is treated with:
- Precise conceptual explanation
- Working code from basic → advanced
- A *WHY THIS MATTERS IN DATA SCIENCE* note
- An *INTERVIEW QUESTION* callout with a model answer

**Prerequisites:** Python Basics notebook. This goes deeper on every topic.

---
### Table of Contents
1. List vs List Comprehension vs Generator Expression
2. Mutable vs Immutable
3. `*args` and `**kwargs`
4. Decorators
5. Generators and `yield`
6. Context Managers
7. Lambda, map, filter, reduce
8. Power Built-ins: zip, enumerate, any, all, sorted
9. Dictionary Tricks
10. String Formatting
11. Exception Handling Deep Dive
12. OOP for Data Science
13. Python Memory Model
14. Underscore Conventions
15. Common Interview Traps

---
## 1. List vs List Comprehension vs Generator Expression

Three ways to produce a sequence of values — each with different **memory profiles** and **performance characteristics**.

| Construct | Syntax | Memory | Reusable | When to use |
|-----------|--------|--------|----------|-------------|
| `list` literal / loop | `[]` | Allocates all at once | Yes | Need indexing / multiple passes |
| List comprehension | `[x for x in ...]` | Allocates all at once | Yes | Readable, idiomatic transformation |
| Generator expression | `(x for x in ...)` | O(1) — lazy | No (exhausted once) | Huge data, pipelines, streaming |

In [1]:
import sys

# --- Basic forms ---
squares_loop = []
for x in range(10):
    squares_loop.append(x ** 2)

squares_comp = [x ** 2 for x in range(10)]        # list comprehension
squares_gen  = (x ** 2 for x in range(10))         # generator expression

print(squares_loop)
print(squares_comp)
print(squares_gen)          # <generator object> — nothing materialised yet
print(list(squares_gen))    # consume it now

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
<generator object <genexpr> at 0xf99b5c15a4d0>
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [2]:
# --- Memory comparison ---
N = 1_000_000

lst  = [x * 2 for x in range(N)]
gen  = (x * 2 for x in range(N))

print(f"List  size: {sys.getsizeof(lst):>12,} bytes")
print(f"Gen   size: {sys.getsizeof(gen):>12,} bytes")   # ~120 bytes regardless of N

List  size:    8,448,728 bytes
Gen   size:          200 bytes


In [3]:
# --- Nested comprehensions ---
# Flatten a 2D list
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat   = [val for row in matrix for val in row]
print("Flat:", flat)

# Transpose a matrix
transposed = [[row[i] for row in matrix] for i in range(3)]
print("Transposed:", transposed)

# Conditional filtering inside comprehension
evens = [x for x in range(20) if x % 2 == 0]
print("Evens:", evens)

Flat: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Transposed: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
Evens: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


In [4]:
# --- Set and dict comprehensions ---
words  = ['apple', 'banana', 'avocado', 'blueberry', 'cherry']
first_letters = {w[0] for w in words}           # set comprehension — unordered, unique
word_lengths  = {w: len(w) for w in words}       # dict comprehension

print("First letters:", first_letters)
print("Lengths:", word_lengths)

First letters: {'a', 'c', 'b'}
Lengths: {'apple': 5, 'banana': 6, 'avocado': 7, 'blueberry': 9, 'cherry': 6}


> **WHY THIS MATTERS IN DATA SCIENCE**
> When processing large CSV files, log streams, or database cursors row-by-row, a generator keeps memory flat. Loading 10 GB of records into a list will OOM your machine; iterating with a generator does not.

> **INTERVIEW QUESTION**
> *"What is the difference between `[x*2 for x in range(100)]` and `(x*2 for x in range(100))`? When would you choose one over the other?"*
>
> **Model answer:** The first is a list comprehension — it evaluates eagerly and stores all 100 values in memory at once. The second is a generator expression — it is lazy; values are produced one at a time only when consumed. Choose a list when you need random access, multiple iterations, or the `len()`. Choose a generator when the dataset is large, you only need one pass, or you are chaining transformations (pipeline). Both have the same time complexity per element; the generator wins on peak memory.

---
## 2. Mutable vs Immutable

**Immutable** objects (int, float, str, tuple, frozenset, bytes) cannot be changed after creation. Python may reuse the same object in memory.

**Mutable** objects (list, dict, set, most class instances) can be modified in place. This has critical implications for default arguments and aliasing bugs.

In [5]:
# --- id() reveals object identity ---
a = 256      # CPython caches small integers -5 to 256
b = 256
print(f"a is b: {a is b}, id(a)={id(a)}, id(b)={id(b)}")

c = 257
d = 257
print(f"c is d: {c is d}")    # May be False outside interactive sessions

# Strings — interning
s1 = 'hello'
s2 = 'hello'
print(f"s1 is s2: {s1 is s2}")   # True — CPython interns short strings

s3 = 'hello world'
s4 = 'hello world'
print(f"s3 is s4: {s3 is s4}")   # May vary

a is b: True, id(a)=11653496, id(b)=11653496
c is d: False
s1 is s2: True
s3 is s4: False


In [6]:
# --- is vs == ---
x = [1, 2, 3]
y = [1, 2, 3]
z = x

print(f"x == y : {x == y}")   # True — same VALUE
print(f"x is y : {x is y}")   # False — different OBJECTS
print(f"x is z : {x is z}")   # True — same OBJECT (alias)

z.append(4)
print(f"x after z.append(4): {x}")   # x is also [1,2,3,4] — mutation through alias!

x == y : True
x is y : False
x is z : True
x after z.append(4): [1, 2, 3, 4]


In [7]:
# --- The mutable default argument trap ---
def add_item_BAD(item, lst=[]):   # lst is created ONCE at function definition time
    lst.append(item)
    return lst

print(add_item_BAD('a'))   # ['a']
print(add_item_BAD('b'))   # ['a', 'b']  <-- SURPRISE! Not ['b']
print(add_item_BAD('c'))   # ['a', 'b', 'c']

# --- Correct idiom ---
def add_item_GOOD(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print(add_item_GOOD('a'))   # ['a']
print(add_item_GOOD('b'))   # ['b']  -- fresh list every call

['a']
['a', 'b']
['a', 'b', 'c']
['a']
['b']


In [8]:
import copy

# --- Shallow copy vs deep copy ---
original = [[1, 2], [3, 4], [5, 6]]

shallow  = original.copy()     # or list(original) or original[:]
deep     = copy.deepcopy(original)

original[0].append(99)   # mutate a nested list

print("original:", original)   # [[1, 2, 99], [3, 4], [5, 6]]
print("shallow: ", shallow)    # [[1, 2, 99], [3, 4], [5, 6]]  <-- affected!
print("deep:    ", deep)       # [[1, 2], [3, 4], [5, 6]]      <-- unaffected

original: [[1, 2, 99], [3, 4], [5, 6]]
shallow:  [[1, 2, 99], [3, 4], [5, 6]]
deep:     [[1, 2], [3, 4], [5, 6]]


> **WHY THIS MATTERS IN DATA SCIENCE**
> Pandas DataFrames are mutable objects. When you do `df2 = df` you get an alias, not a copy. Modifying `df2` changes `df`. Always use `df.copy()` when you need an independent DataFrame. Similarly, `sklearn` transformers are stateful mutable objects — fitting them mutates internal state.

> **INTERVIEW QUESTION**
> *"Why is using a list as a default argument dangerous in Python? How do you fix it?"*
>
> **Model answer:** Default argument values are evaluated **once** when the `def` statement executes, not on every call. If the default is a mutable object like `[]`, all calls that rely on the default share the **same** object. Mutations in one call persist into the next. The fix is to use `None` as the sentinel and create a fresh list inside the function body with `if lst is None: lst = []`.

---
## 3. `*args` and `**kwargs`

`*args` collects extra positional arguments into a **tuple**.
`**kwargs` collects extra keyword arguments into a **dict**.
They are just naming conventions — `*` and `**` are the syntax.

In [9]:
# --- Basic *args ---
def total(*args):
    print(f"args type: {type(args)}, values: {args}")
    return sum(args)

print(total(1, 2, 3))          # 6
print(total(10, 20, 30, 40))   # 100

args type: <class 'tuple'>, values: (1, 2, 3)
6
args type: <class 'tuple'>, values: (10, 20, 30, 40)
100


In [10]:
# --- Basic **kwargs ---
def show_config(**kwargs):
    print(f"kwargs type: {type(kwargs)}")
    for key, val in kwargs.items():
        print(f"  {key} = {val}")

show_config(learning_rate=0.01, n_estimators=100, max_depth=5)

kwargs type: <class 'dict'>
  learning_rate = 0.01
  n_estimators = 100
  max_depth = 5


In [11]:
# --- Combining positional, *args, keyword-only, **kwargs ---
# Order MUST be: positional -> *args -> keyword-only -> **kwargs

def pipeline(data, *transforms, verbose=False, **fit_params):
    """Apply a sequence of transforms with optional fit parameters."""
    if verbose:
        print(f"Running {len(transforms)} transforms with params: {fit_params}")
    result = data
    for fn in transforms:
        result = fn(result)
    return result

import math
result = pipeline(
    [1, 4, 9, 16],
    lambda x: [v * 2 for v in x],
    lambda x: [math.sqrt(v) for v in x],
    verbose=True,
    scale=1.0
)
print(result)

Running 2 transforms with params: {'scale': 1.0}
[1.4142135623730951, 2.8284271247461903, 4.242640687119285, 5.656854249492381]


In [12]:
# --- Unpacking with * and ** at call site ---
def train_model(X, y, lr=0.01, epochs=10):
    print(f"Training: lr={lr}, epochs={epochs}, X.shape={len(X)}x{len(X[0])}")

args_list = ([[1,2],[3,4],[5,6]], [0, 1, 0])
kwargs_dict = {'lr': 0.001, 'epochs': 50}

train_model(*args_list, **kwargs_dict)   # equivalent to train_model(X, y, lr=0.001, epochs=50)

Training: lr=0.001, epochs=50, X.shape=3x2


In [13]:
# --- Decorator pattern using *args/**kwargs ---
import time
import functools

def timer(func):
    """Decorator that measures execution time of any function."""
    @functools.wraps(func)   # preserve original function metadata
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)
        end    = time.perf_counter()
        print(f"{func.__name__!r} took {(end-start)*1000:.3f} ms")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(1_000_000)
print(f"Result: {result}")

'slow_sum' took 39.389 ms
Result: 499999500000


> **WHY THIS MATTERS IN DATA SCIENCE**
> `**kwargs` is the backbone of scikit-learn's `set_params()` API and pandas' `DataFrame.to_csv(**kwargs)`. When writing wrapper functions or pipelines that pass parameters through to underlying libraries, `**kwargs` lets your wrapper stay forward-compatible as the library evolves.

> **INTERVIEW QUESTION**
> *"Explain the difference between `*args` and `**kwargs`. What is the order rule when combining them?"*
>
> **Model answer:** `*args` captures extra positional arguments as a tuple; `**kwargs` captures extra keyword arguments as a dict. The required ordering in a signature is: regular positional params → `*args` → keyword-only params → `**kwargs`. This ordering is enforced by Python's parser. At the call site, `*iterable` unpacks any iterable as positional args and `**mapping` unpacks any dict as keyword args.

---
## 4. Decorators

A decorator is a **higher-order function** — a callable that takes a function and returns a modified version of it. The `@decorator` syntax is pure syntactic sugar for `func = decorator(func)`.

In [14]:
import functools

# --- Building a decorator from scratch ---
def log_calls(func):
    @functools.wraps(func)  # copies __name__, __doc__, __annotations__ to wrapper
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} returned {result}")
        return result
    return wrapper

@log_calls
def add(x, y):
    """Add two numbers."""
    return x + y

add(3, 5)
print("Function name preserved:", add.__name__)   # 'add', not 'wrapper'

[LOG] Calling add with args=(3, 5), kwargs={}
[LOG] add returned 8
Function name preserved: add


In [15]:
# --- Decorator with arguments (factory pattern) ---
def retry(max_attempts=3, exceptions=(Exception,)):
    """Retry a function up to max_attempts times on specified exceptions."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

@retry(max_attempts=3, exceptions=(ValueError,))
def flaky_parse(value):
    import random
    if random.random() < 0.5:
        raise ValueError("Parse failed")
    return int(value)

try:
    result = flaky_parse('42')
    print(f"Got: {result}")
except ValueError:
    print("All retries exhausted")

Got: 42

In [16]:
# --- @property, @staticmethod, @classmethod in a DS context ---

class DatasetStats:
    _registry = []   # class-level storage

    def __init__(self, data: list, name: str):
        self._data = data
        self.name  = name
        DatasetStats._registry.append(name)

    # --- @property: computed attribute, no () needed at call site ---
    @property
    def mean(self):
        return sum(self._data) / len(self._data)

    @property
    def data(self):
        return self._data

    @data.setter
    def data(self, new_data):
        if not isinstance(new_data, list):
            raise TypeError("data must be a list")
        self._data = new_data

    # --- @staticmethod: utility, no self/cls, logically belongs to class ---
    @staticmethod
    def normalize(values):
        mn, mx = min(values), max(values)
        return [(v - mn) / (mx - mn) for v in values]

    # --- @classmethod: factory / alternative constructor ---
    @classmethod
    def from_csv_row(cls, row_string, name):
        data = [float(x) for x in row_string.split(',')]
        return cls(data, name)

    @classmethod
    def list_datasets(cls):
        return cls._registry

# Usage
ds1 = DatasetStats([10, 20, 30, 40, 50], 'train')
ds2 = DatasetStats.from_csv_row('1.5,2.5,3.5,4.5', 'test')

print(f"Mean: {ds1.mean}")                                # property — no ()
print(f"Normalized: {DatasetStats.normalize([10,20,30])}") # staticmethod
print(f"All datasets: {DatasetStats.list_datasets()}")    # classmethod

Mean: 30.0
Normalized: [0.0, 0.5, 1.0]
All datasets: ['train', 'test']


> **WHY THIS MATTERS IN DATA SCIENCE**
> Decorators are everywhere in ML code: `@tf.function` (TensorFlow), `@torch.no_grad()` (PyTorch), `@pytest.fixture`. Understanding the pattern lets you write caching (`@functools.lru_cache`), timing, and validation decorators for your own pipelines. `@property` is how pandas exposes `df.shape`, `df.dtypes`, etc.

> **INTERVIEW QUESTION**
> *"What is the difference between `@staticmethod` and `@classmethod`? Give a concrete use case for each."*
>
> **Model answer:** A `@staticmethod` receives neither the instance nor the class — it is a plain function namespaced inside the class. Use it for pure utility logic that conceptually belongs to the class (e.g., `normalize`). A `@classmethod` receives `cls` as the first argument — it can access and modify class-level state and is the canonical way to write alternative constructors (`from_csv`, `from_dict`). The difference matters when subclassing: `cls` in a classmethod refers to the actual subclass, enabling polymorphic factories.

---
## 5. Generators and `yield`

A **generator function** uses `yield` to produce values lazily. Each call to `next()` resumes execution from where it last `yield`ed. The function's local state (variables, instruction pointer) is frozen between yields.

In [17]:
# --- yield basics ---
def countdown(n):
    print("Starting countdown")
    while n > 0:
        yield n          # execution pauses here; state is saved
        n -= 1
    print("Countdown done")

gen = countdown(3)      # nothing runs yet
print(type(gen))
print(next(gen))        # prints "Starting countdown", returns 3
print(next(gen))        # returns 2
print(next(gen))        # returns 1
# next(gen) here would raise StopIteration and print "Countdown done"

<class 'generator'>
Starting countdown
3
2
1


In [18]:
# --- range is like a generator (but is actually a sequence type) ---
# The key insight: both avoid materialising all values at once
import sys

r = range(1_000_000)
l = list(range(1_000_000))
print(f"range size: {sys.getsizeof(r)} bytes")
print(f"list  size: {sys.getsizeof(l):,} bytes")

range size: 48 bytes
list  size: 8,000,056 bytes


In [19]:
# --- Generator pipeline — lazy ETL ---
def read_lines(filename):
    """Simulate reading a large file line by line."""
    fake_lines = [
        "alice,30,engineer",
        "bob,25,analyst",
        "charlie,35,scientist",
        "diana,28,engineer",
    ]
    for line in fake_lines:
        yield line

def parse_csv(lines):
    for line in lines:
        parts = line.split(',')
        yield {'name': parts[0], 'age': int(parts[1]), 'role': parts[2]}

def filter_engineers(records):
    for r in records:
        if r['role'] == 'engineer':
            yield r

# Chain generators — nothing runs until we consume
pipeline = filter_engineers(parse_csv(read_lines('data.csv')))

for record in pipeline:
    print(record)    # only engineers, one at a time, O(1) memory

{'name': 'alice', 'age': 30, 'role': 'engineer'}
{'name': 'diana', 'age': 28, 'role': 'engineer'}


In [20]:
# --- yield from (Python 3.3+) — delegating to sub-generators ---
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # delegate recursively
        else:
            yield item

data = [1, [2, 3, [4, 5]], 6, [7, [8, 9]]]
print(list(flatten(data)))

[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [21]:
# --- send() — two-way communication with generators ---
def running_average():
    total, count = 0, 0
    while True:
        value = yield (total / count if count else None)
        total += value
        count += 1

avg = running_average()
next(avg)                         # prime the generator
print(avg.send(10))               # 10.0
print(avg.send(20))               # 15.0
print(avg.send(30))               # 20.0

10.0
15.0
20.0


> **WHY THIS MATTERS IN DATA SCIENCE**
> Keras/TensorFlow's `model.fit()` accepts a Python generator as `x` — the generator pattern is the official way to feed batches of training data that don't fit in RAM. `itertools` is built on generators. `pandas.read_csv(chunksize=N)` returns a generator of DataFrames.

> **INTERVIEW QUESTION**
> *"How does a generator function differ from a regular function? What happens to its state between `yield` calls?"*
>
> **Model answer:** Calling a generator function does not execute it — it returns a generator object. Execution begins on the first `next()` call and runs until the first `yield`, which suspends the frame. The function's entire local state (variables, instruction pointer, stack) is preserved on the heap. The next `next()` resumes from exactly after the `yield`. This makes generators ideal for representing infinite sequences or streaming pipelines without holding all values in memory simultaneously.

---
## 6. Context Managers

A context manager guarantees that setup and teardown happen in a pair — even if exceptions occur. The `with` statement calls `__enter__` on entry and `__exit__` on exit.

In [22]:
# --- Classic: file handling ---
# Bad: file may not be closed if exception occurs
# f = open('data.txt'); data = f.read(); f.close()

# Good: __exit__ closes even on exception
# with open('data.txt', 'r') as f:
#     data = f.read()

# --- Write a custom context manager using a class ---
import time

class Timer:
    """Context manager that measures block execution time."""

    def __enter__(self):
        self.start = time.perf_counter()
        return self           # value bound to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed * 1000:.3f} ms")
        return False          # False = do NOT suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))

print(f"Total: {total}, time stored: {t.elapsed:.4f}s")

Elapsed: 53.604 ms
Total: 499999500000, time stored: 0.0536s


In [23]:
# --- contextlib.contextmanager — generator-based (much simpler) ---
from contextlib import contextmanager

@contextmanager
def temp_numpy_config(**kwargs):
    """Temporarily change numpy print options and restore them."""
    import numpy as np
    old_options = np.get_printoptions()
    np.set_printoptions(**kwargs)
    try:
        yield          # code inside 'with' block runs here
    finally:
        np.set_printoptions(**old_options)   # always restore

import numpy as np
arr = np.array([1.123456789, 2.987654321, 3.14159265])
print("Default:", arr)
with temp_numpy_config(precision=2, suppress=True):
    print("Inside context:", arr)
print("After context:", arr)

Default: [1.12345679 2.98765432 3.14159265]
Inside context: [1.12 2.99 3.14]
After context: [1.12345679 2.98765432 3.14159265]


In [24]:
# --- contextlib.suppress — swallow specific exceptions ---
from contextlib import suppress

data = {'a': 1, 'b': 2}

with suppress(KeyError):
    val = data['missing_key']   # KeyError is silently suppressed
    print("This won't print")

print("Execution continues normally here")

Execution continues normally here


In [25]:
# --- Exception suppression control via __exit__ return value ---
class SuppressValueError:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is ValueError:
            print(f"Suppressing ValueError: {exc_val}")
            return True    # True suppresses the exception
        return False       # propagate anything else

with SuppressValueError():
    raise ValueError("bad input")   # suppressed!

print("After suppressed exception")

Suppressing ValueError: bad input
After suppressed exception


> **WHY THIS MATTERS IN DATA SCIENCE**
> Database connections, file handles, GPU memory locks, and distributed locks all need guaranteed cleanup. `torch.no_grad()` and `tf.device(...)` are context managers. Writing your own lets you safely manage resources in notebooks that often get interrupted or restarted.

> **INTERVIEW QUESTION**
> *"What does `__exit__` returning `True` vs `False` mean?"*
>
> **Model answer:** `__exit__` receives three arguments describing any exception (`exc_type`, `exc_val`, `exc_tb`). If `__exit__` returns a truthy value, Python suppresses the exception and execution continues after the `with` block. If it returns a falsy value (or `None`), any exception propagates normally. This is how `contextlib.suppress()` works — its `__exit__` returns `True` when the exception type matches the suppression list.

---
## 7. Lambda, map, filter, reduce

`lambda` creates anonymous single-expression functions. `map`, `filter`, and `reduce` apply functions over iterables. In modern Python, comprehensions are usually preferred for readability, but these are still common in interviews and functional-style code.

In [26]:
# --- lambda basics ---
square = lambda x: x ** 2
add    = lambda x, y: x + y
clamp  = lambda x, lo, hi: max(lo, min(hi, x))

print(square(5))
print(add(3, 4))
print(clamp(15, 0, 10))   # 10

25
7
10


In [27]:
# --- map: apply function to each element, returns iterator ---
data = [1, 2, 3, 4, 5]

doubled_map  = list(map(lambda x: x * 2, data))
doubled_comp = [x * 2 for x in data]           # equivalent, more Pythonic

print(doubled_map)
print(doubled_comp)

# map with two iterables
a = [1, 2, 3]
b = [10, 20, 30]
sums = list(map(lambda x, y: x + y, a, b))
print(sums)   # [11, 22, 33]

[2, 4, 6, 8, 10]
[2, 4, 6, 8, 10]
[11, 22, 33]


In [28]:
# --- filter: keep elements where predicate is True ---
scores = [45, 72, 88, 33, 95, 61, 77]

passing_filter = list(filter(lambda x: x >= 60, scores))
passing_comp   = [x for x in scores if x >= 60]          # equivalent

print(passing_filter)
print(passing_comp)

[72, 88, 95, 61, 77]
[72, 88, 95, 61, 77]


In [29]:
from functools import reduce

# --- reduce: fold a sequence to a single value ---
nums = [1, 2, 3, 4, 5]
product  = reduce(lambda acc, x: acc * x, nums)       # 120
max_val  = reduce(lambda a, b: a if a > b else b, nums)  # 5

print(f"Product: {product}")
print(f"Max via reduce: {max_val}")

# With initializer
total_with_bonus = reduce(lambda acc, x: acc + x, nums, 100)   # 100 + 15 = 115
print(f"Sum with bonus: {total_with_bonus}")

Product: 120
Max via reduce: 5
Sum with bonus: 115


In [30]:
# --- When lambda wins: sort keys, callbacks ---
records = [
    {'name': 'Alice', 'score': 88, 'age': 30},
    {'name': 'Bob',   'score': 95, 'age': 25},
    {'name': 'Carol', 'score': 88, 'age': 28},
]

# Sort by score descending, then age ascending
sorted_records = sorted(records, key=lambda r: (-r['score'], r['age']))
for r in sorted_records:
    print(r)

{'name': 'Bob', 'score': 95, 'age': 25}


{'name': 'Carol', 'score': 88, 'age': 28}
{'name': 'Alice', 'score': 88, 'age': 30}


> **WHY THIS MATTERS IN DATA SCIENCE**
> `df.apply(lambda x: ...)` is how you apply custom logic column/row-wise in pandas. `sorted(..., key=lambda ...)` is ubiquitous. `map` with built-in functions like `str`, `int`, `float` is efficient and common in data cleaning.

> **INTERVIEW QUESTION**
> *"When would you choose `map()` over a list comprehension?"*
>
> **Model answer:** In modern Python 3, list comprehensions are almost always more readable and roughly equivalent in performance. `map()` has an edge when you already have a named function (avoids creating a lambda): `map(str, numbers)` vs `[str(n) for n in numbers]` — the former is marginally faster. `map` is also lazy (returns an iterator), so it composes well with other lazy operations. In data science practice, comprehensions win on readability; `map` with a named function wins in tight inner loops.

---
## 8. Power Built-ins: zip, enumerate, any, all, sorted

Python's built-in functions are written in C and are extremely fast. Knowing them deeply is a mark of Python fluency.

In [31]:
# --- zip: pair up multiple iterables ---
features = ['age', 'salary', 'experience']
values   = [30, 75000, 7]

paired = dict(zip(features, values))
print(paired)

# zip stops at shortest — zip_longest preserves all
from itertools import zip_longest
a = [1, 2, 3]
b = [10, 20]
print(list(zip(a, b)))                       # [(1,10),(2,20)]
print(list(zip_longest(a, b, fillvalue=0)))  # [(1,10),(2,20),(3,0)]

# Unzip (transpose) with *
points = [(1, 4), (2, 5), (3, 6)]
xs, ys = zip(*points)
print(f"xs={xs}, ys={ys}")

{'age': 30, 'salary': 75000, 'experience': 7}
[(1, 10), (2, 20)]
[(1, 10), (2, 20), (3, 0)]
xs=(1, 2, 3), ys=(4, 5, 6)


In [32]:
# --- enumerate: index + value ---
classes = ['cat', 'dog', 'bird']

# Idiomatic — never use range(len(...))
for idx, cls in enumerate(classes, start=1):  # start parameter!
    print(f"{idx}. {cls}")

# Build index lookup
label_to_idx = {cls: idx for idx, cls in enumerate(classes)}
print(label_to_idx)

1. cat
2. dog
3. bird
{'cat': 0, 'dog': 1, 'bird': 2}


In [33]:
# --- any() and all() — short-circuit evaluation ---
predictions = [0.9, 0.3, 0.8, 0.95]
labels      = [1, 0, None, 1]

# any: True if at least one element is truthy
has_null  = any(v is None for v in labels)
high_conf = any(p > 0.9 for p in predictions)
print(f"Has null: {has_null}, High confidence: {high_conf}")

# all: True only if every element is truthy
all_valid    = all(v is not None for v in labels)
all_positive = all(p > 0 for p in predictions)
print(f"All valid: {all_valid}, All positive: {all_positive}")

# Empty sequences
print(f"any([]) = {any([])}")   # False
print(f"all([]) = {all([])}")   # True (vacuously)

Has null: True, High confidence: True
All valid: False, All positive: True
any([]) = False
all([]) = True


In [34]:
# --- sorted with key= ---
models = [
    ('RandomForest', 0.923, 0.031),
    ('XGBoost',      0.941, 0.018),
    ('LogReg',       0.872, 0.005),
    ('SVM',          0.910, 0.022),
]

# Sort by accuracy descending
by_acc = sorted(models, key=lambda m: m[1], reverse=True)
print("By accuracy:", [(m[0], m[1]) for m in by_acc])

# Sort by training time ascending
by_time = sorted(models, key=lambda m: m[2])
print("By training time:", [(m[0], m[2]) for m in by_time])

# operator.itemgetter is faster than lambda for simple key access
from operator import itemgetter
by_time2 = sorted(models, key=itemgetter(2))
print("Using itemgetter:", [(m[0], m[2]) for m in by_time2])

By accuracy: [('XGBoost', 0.941), ('RandomForest', 0.923), ('SVM', 0.91), ('LogReg', 0.872)]
By training time: [('LogReg', 0.005), ('XGBoost', 0.018), ('SVM', 0.022), ('RandomForest', 0.031)]
Using itemgetter: [('LogReg', 0.005), ('XGBoost', 0.018), ('SVM', 0.022), ('RandomForest', 0.031)]


In [35]:
# --- Tricky interview questions about these built-ins ---

# Q: What does zip return in Python 3?
z = zip([1,2,3], ['a','b','c'])
print(type(z))        # <class 'zip'> — an iterator, NOT a list

# Q: Can you reuse a zip object?
first_pass  = list(z)
second_pass = list(z)  # already exhausted!
print(f"First: {first_pass}")
print(f"Second: {second_pass}")   # []

# Q: What does all([]) return and why?
print(f"all([]) = {all([])}")  # True — vacuous truth (no counterexamples exist)

<class 'zip'>
First: [(1, 'a'), (2, 'b'), (3, 'c')]
Second: []
all([]) = True


> **WHY THIS MATTERS IN DATA SCIENCE**
> `zip` is the Pythonic way to iterate feature names and values together. `enumerate` replaces `range(len(...))`. `any`/`all` with generator expressions short-circuit — they stop as soon as the answer is known, making them efficient for data validation checks over large arrays.

> **INTERVIEW QUESTION**
> *"What is the difference between `sorted()` and `.sort()`? What does the `key` parameter do?"*
>
> **Model answer:** `.sort()` is a list method that sorts in-place and returns `None`. `sorted()` is a built-in that works on any iterable and returns a new list, leaving the original unchanged. The `key` parameter takes a callable that is applied to each element before comparison — only the return value is compared, not the element itself. This avoids creating a temporary list of `(key, element)` tuples manually. Python's sort is stable (Timsort), so equal keys preserve original order.

---
## 9. Dictionary Tricks

Dictionaries are central to Python. The `collections` module provides powerful specialisations. Mastering dict operations is essential for feature engineering, counting, grouping, and config management.

In [36]:
from collections import defaultdict, Counter, OrderedDict

# --- defaultdict: never get KeyError for missing keys ---
# Grouping by category
records = [
    ('Alice', 'Engineering'), ('Bob', 'Marketing'), ('Carol', 'Engineering'),
    ('Dave', 'Marketing'), ('Eve', 'Engineering')
]

groups = defaultdict(list)
for name, dept in records:
    groups[dept].append(name)   # no KeyError — list() is called on first access

print(dict(groups))

{'Engineering': ['Alice', 'Carol', 'Eve'], 'Marketing': ['Bob', 'Dave']}


In [37]:
# --- Counter: count hashable objects ---
words = "the quick brown fox jumps over the lazy dog the fox".split()
counts = Counter(words)

print(counts)                          # Counter({'the': 3, 'fox': 2, ...})
print(counts.most_common(3))           # [(word, count)] top-3
print(counts['missing_word'])          # 0, not KeyError!

# Counter arithmetic
more = Counter({'the': 5, 'cat': 3})
print(counts + more)                   # element-wise addition
print(counts - more)                   # subtraction, drops zero/negative

Counter({'the': 3, 'fox': 2, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})
[('the', 3), ('fox', 2), ('quick', 1)]
0
Counter({'the': 8, 'cat': 3, 'fox': 2, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})
Counter({'fox': 2, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})


In [38]:
# --- .get() vs [] — safe access ---
config = {'lr': 0.01, 'epochs': 100}

# Bad: raises KeyError if key absent
# val = config['batch_size']

# Good: returns default
batch_size = config.get('batch_size', 32)   # 32 if missing
print(f"batch_size: {batch_size}")

# setdefault: get value, and INSERT if missing
config.setdefault('dropout', 0.5)
print(config)

batch_size: 32
{'lr': 0.01, 'epochs': 100, 'dropout': 0.5}


In [39]:
# --- Merging dicts ---
defaults = {'lr': 0.01, 'epochs': 100, 'batch_size': 32}
overrides = {'lr': 0.001, 'epochs': 50}

# Python 3.9+: | operator (creates new dict)
merged = defaults | overrides
print("| merge:", merged)

# Python 3.9+: |= in-place merge
d = defaults.copy()
d |= overrides
print("|= merge:", d)

# Python 3.5+: **-unpacking (works everywhere)
merged2 = {**defaults, **overrides}   # right side wins on conflict
print("** merge:", merged2)

| merge: {'lr': 0.001, 'epochs': 50, 'batch_size': 32}
|= merge: {'lr': 0.001, 'epochs': 50, 'batch_size': 32}
** merge: {'lr': 0.001, 'epochs': 50, 'batch_size': 32}


In [40]:
# --- Dict comprehension for data transformation ---
raw_features = {'age': '30', 'salary': '75000', 'experience': '7'}

# Cast all values to float
numeric = {k: float(v) for k, v in raw_features.items()}
print(numeric)

# Invert a dict (value -> key)
label_map = {'cat': 0, 'dog': 1, 'bird': 2}
inv_map   = {v: k for k, v in label_map.items()}
print(inv_map)

# Filter keys
important = {k: v for k, v in numeric.items() if v > 10}
print(important)

{'age': 30.0, 'salary': 75000.0, 'experience': 7.0}
{0: 'cat', 1: 'dog', 2: 'bird'}
{'age': 30.0, 'salary': 75000.0}


In [41]:
# --- OrderedDict: mostly historical (dicts are ordered in Python 3.7+) ---
# Still useful for its move_to_end() and explicit ordering semantics
from collections import OrderedDict

od = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
od.move_to_end('a')          # move 'a' to end
od.move_to_end('c', last=False)  # move 'c' to front
print(od)

# OrderedDict equality checks order; regular dicts don't
d1 = OrderedDict([('a', 1), ('b', 2)])
d2 = OrderedDict([('b', 2), ('a', 1)])
print(f"OrderedDict equal: {d1 == d2}")    # False
print(f"Regular dict equal: {dict(d1) == dict(d2)}")   # True

OrderedDict({'c': 3, 'b': 2, 'a': 1})
OrderedDict equal: False
Regular dict equal: True


> **WHY THIS MATTERS IN DATA SCIENCE**
> `Counter` is the fastest way to compute value frequencies for EDA. `defaultdict(list)` is the canonical groupby before you reach for pandas. Dict comprehensions replace many `for`-loop data transformations. The `|` merge operator cleanly handles hyperparameter config layering (defaults merged with user overrides).

> **INTERVIEW QUESTION**
> *"What is the time complexity of dict lookup, insertion, and deletion? How does a dict handle hash collisions?"*
>
> **Model answer:** All three are O(1) average case, O(n) worst case. Python dicts use open addressing with a probe sequence. On collision, Python probes other slots using a pseudorandom sequence derived from the hash. The dict resizes (rehashing all entries) when the load factor exceeds ~2/3. Python's hash table is compact since Python 3.6 and maintains insertion order since 3.7.

---
## 10. String Formatting

Python has three formatting systems: `%` (old), `.format()` (Python 3.0), and f-strings (Python 3.6+). Know all three for maintenance work and interviews.

In [42]:
name   = "Alice"
score  = 0.9235
epochs = 1000

# --- % formatting (C-style, legacy) ---
print("Name: %s, Score: %.2f, Epochs: %d" % (name, score, epochs))

# --- str.format() ---
print("Name: {}, Score: {:.2f}, Epochs: {:,}".format(name, score, epochs))
print("Name: {n}, Score: {s:.4f}".format(n=name, s=score))   # named

# --- f-strings (Python 3.6+) — preferred ---
print(f"Name: {name}, Score: {score:.2f}, Epochs: {epochs:,}")

Name: Alice, Score: 0.92, Epochs: 1000
Name: Alice, Score: 0.92, Epochs: 1,000
Name: Alice, Score: 0.9235
Name: Alice, Score: 0.92, Epochs: 1,000


In [43]:
# --- f-string superpowers ---

# Arbitrary expressions
x = 42
print(f"x squared = {x**2}")
print(f"Is even: {x % 2 == 0}")

# Calling functions inside f-strings
data = [3, 1, 4, 1, 5, 9]
print(f"Max: {max(data)}, Sorted: {sorted(data)}")

# Format spec expressions (Python 3.12+ supports nested f-strings fully)
precision = 3
print(f"Score: {score:.{precision}f}")

# = specifier (Python 3.8+) — great for debugging
model_acc = 0.9235
print(f"{model_acc = :.3f}")   # prints: model_acc = 0.924

# !r, !s, !a conversions
raw = "hello\nworld"
print(f"!s: {raw!s}")   # str()  — normal string
print(f"!r: {raw!r}")   # repr() — escaped string with quotes

x squared = 1764
Is even: True
Max: 9, Sorted: [1, 1, 3, 4, 5, 9]
Score: 0.923
model_acc = 0.923
!s: hello
world
!r: 'hello\nworld'


In [44]:
# --- Format spec mini-language ---
pi = 3.14159265

print(f"{pi:10.4f}")    # width=10, 4 decimal places, right-aligned (default)
print(f"{pi:<10.4f}")   # left-aligned
print(f"{pi:^10.4f}")   # center-aligned
print(f"{pi:0>10.4f}")  # zero-filled right-align
print(f"{1234567:,}")   # thousands separator
print(f"{0.875:.1%}")   # percentage
print(f"{255:08b}")     # binary with zero-padding
print(f"{255:02x}")     # hexadecimal

    3.1416


3.1416    
  3.1416  
00003.1416
1,234,567
87.5%
11111111
ff


In [45]:
# --- Performance: f-strings are fastest ---
import timeit

setup = "name='Alice'; score=0.9235"
n = 100_000

t1 = timeit.timeit("'Name: %s, Score: %.2f' % (name, score)", setup=setup, number=n)
t2 = timeit.timeit("'Name: {}, Score: {:.2f}'.format(name, score)", setup=setup, number=n)
t3 = timeit.timeit("f'Name: {name}, Score: {score:.2f}'", setup=setup, number=n)

print(f"% format:      {t1*1000:.1f}ms")
print(f".format():     {t2*1000:.1f}ms")
print(f"f-string:      {t3*1000:.1f}ms")

% format:      37.3ms


.format():     43.8ms
f-string:      38.9ms


> **WHY THIS MATTERS IN DATA SCIENCE**
> Clear formatted output in training loops, metric reports, and logging matters for debugging. The `=` specifier in f-strings is invaluable during notebook development. Understanding format specs lets you produce clean tabular output without pandas.

> **INTERVIEW QUESTION**
> *"What does `f"{value!r}"` do? When would you use `!r` vs `!s`?"*
>
> **Model answer:** `!r` applies `repr()` to the value before formatting — this is the unambiguous, developer-facing representation. For strings it adds quotes and escapes special characters. `!s` applies `str()` — the human-readable form. Use `!r` when you want to log a value in a form that could be copy-pasted back as Python code (debugging), and `!s` (the default) for end-user messages.

---
## 11. Exception Handling Deep Dive

Exception handling is more nuanced than just `try/except`. Understanding the full model — `else`, `finally`, exception chaining, and custom exceptions — is what separates production-quality code.

In [46]:
# --- Full try/except/else/finally structure ---
def load_data(filename):
    try:
        f = open(filename)
        data = f.read()
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return None
    except PermissionError as e:
        print(f"No permission: {e}")
        return None
    except (IOError, OSError) as e:       # catching multiple exceptions
        print(f"IO error: {e}")
        return None
    else:
        # Runs ONLY if no exception was raised in try block
        print("File loaded successfully")
        return data
    finally:
        # ALWAYS runs — exception or not, return or not
        print("Cleanup (always runs)")

result = load_data('nonexistent.csv')

File not found: [Errno 2] No such file or directory: 'nonexistent.csv'
Cleanup (always runs)


In [47]:
# --- else vs finally: the key distinction ---
def demonstrate_else_finally():
    try:
        result = 10 / 2
    except ZeroDivisionError:
        print("except: division by zero")
    else:
        print(f"else: success, result={result}")   # only if NO exception
    finally:
        print("finally: always executes")           # always, even with return/continue/break

demonstrate_else_finally()

else: success, result=5.0
finally: always executes


In [48]:
# --- Custom exceptions ---
class DataValidationError(ValueError):
    """Raised when input data fails validation checks."""
    def __init__(self, message, field=None, value=None):
        super().__init__(message)
        self.field = field
        self.value = value

    def __str__(self):
        base = super().__str__()
        if self.field:
            return f"{base} (field={self.field!r}, value={self.value!r})"
        return base

class ModelNotFittedError(RuntimeError):
    """Raised when predict() is called before fit()."""
    pass

def validate_age(age):
    if not isinstance(age, (int, float)):
        raise DataValidationError("Age must be numeric", field='age', value=age)
    if age < 0 or age > 150:
        raise DataValidationError("Age out of range [0, 150]", field='age', value=age)
    return age

try:
    validate_age(-5)
except DataValidationError as e:
    print(f"Validation failed: {e}")
    print(f"  field={e.field}, value={e.value}")

Validation failed: Age out of range [0, 150] (field='age', value=-5)
  field=age, value=-5


In [49]:
# --- Exception chaining: raise X from Y ---
def parse_config(raw):
    try:
        return int(raw)
    except ValueError as e:
        raise DataValidationError(f"Cannot parse config value: {raw!r}") from e
        # 'from e' sets __cause__ and shows both exceptions in traceback

try:
    parse_config('abc')
except DataValidationError as e:
    print(f"Caught: {e}")
    print(f"Caused by: {e.__cause__}")

Caught: Cannot parse config value: 'abc'
Caused by: invalid literal for int() with base 10: 'abc'


In [50]:
# --- Bare raise: re-raise current exception ---
def process_batch(batch):
    try:
        result = [1 / x for x in batch]
        return result
    except ZeroDivisionError:
        print("Warning: zero in batch, logging and re-raising")
        raise   # re-raises the same exception with full traceback intact

try:
    process_batch([2, 1, 0, 4])
except ZeroDivisionError as e:
    print(f"Outer handler caught: {e}")

Outer handler caught: division by zero


> **WHY THIS MATTERS IN DATA SCIENCE**
> Production ML pipelines need robust error handling: parse errors in features, missing files, corrupted batches. Custom exceptions with `field` and `value` attributes make debugging much faster. `raise X from Y` preserves the full causal chain in logs. The `else` clause avoids catching exceptions from error-handling code itself.

> **INTERVIEW QUESTION**
> *"What is the difference between `try/except/else/finally`? When does `else` run vs `finally`?"*
>
> **Model answer:** `else` runs if and only if the `try` block completed without raising any exception. `finally` always runs — even if an exception was raised, even if `return`/`break`/`continue` was hit inside `try` or `except`. The `else` clause is useful for code that should only run on success but should NOT be wrapped in `try` (so its own exceptions aren't accidentally swallowed by the existing `except` handlers).

---
## 12. OOP for Data Science

Understanding Python's data model (dunder methods) lets you build objects that integrate seamlessly with the language — iteration, indexing, `len()`, string representation.

In [51]:
class Dataset:
    """
    A minimal dataset class demonstrating Python's data model.
    Supports len(), iteration, indexing, and method chaining.
    """

    def __init__(self, data: list, name: str = 'dataset'):
        self._data = list(data)
        self.name  = name

    # --- __repr__: unambiguous representation for developers ---
    def __repr__(self):
        return f"Dataset(name={self.name!r}, n={len(self._data)})"

    # --- __str__: human-readable string ---
    def __str__(self):
        preview = self._data[:3]
        more    = '...' if len(self._data) > 3 else ''
        return f"Dataset '{self.name}': {preview}{more}"

    # --- __len__: enables len(dataset) ---
    def __len__(self):
        return len(self._data)

    # --- __getitem__: enables dataset[i] and dataset[a:b] ---
    def __getitem__(self, idx):
        return self._data[idx]

    # --- __contains__: enables 'x in dataset' ---
    def __contains__(self, item):
        return item in self._data

    # --- __iter__: enables for x in dataset ---
    def __iter__(self):
        return iter(self._data)

    # --- __eq__: enables dataset1 == dataset2 ---
    def __eq__(self, other):
        if not isinstance(other, Dataset):
            return NotImplemented
        return self._data == other._data

    # --- Method chaining (like pandas) ---
    def filter(self, predicate):
        return Dataset([x for x in self._data if predicate(x)], name=self.name)

    def transform(self, fn):
        return Dataset([fn(x) for x in self._data], name=self.name)

    def head(self, n=5):
        return Dataset(self._data[:n], name=self.name)

# Usage
ds = Dataset(range(20), name='train')
print(repr(ds))               # __repr__
print(str(ds))                # __str__
print(f"Length: {len(ds)}")  # __len__
print(f"ds[0]: {ds[0]}")     # __getitem__
print(f"5 in ds: {5 in ds}") # __contains__

# Method chaining
result = (ds
    .filter(lambda x: x % 2 == 0)   # keep evens
    .transform(lambda x: x ** 2)     # square
    .head(5)                         # first 5
)
print(f"Chained result: {list(result)}")

Dataset(name='train', n=20)
Dataset 'train': [0, 1, 2]...
Length: 20
ds[0]: 0
5 in ds: True
Chained result: [0, 4, 16, 36, 64]


In [52]:
# --- __add__, __mul__ for numeric types ---
class Vector:
    def __init__(self, *components):
        self.data = components

    def __repr__(self):
        return f"Vector{self.data}"

    def __add__(self, other):
        return Vector(*(a + b for a, b in zip(self.data, other.data)))

    def __mul__(self, scalar):
        return Vector(*(x * scalar for x in self.data))

    def __rmul__(self, scalar):   # scalar * vector
        return self.__mul__(scalar)

    def dot(self, other):
        return sum(a * b for a, b in zip(self.data, other.data))

v1 = Vector(1, 2, 3)
v2 = Vector(4, 5, 6)
print(v1 + v2)
print(v1 * 3)
print(3 * v1)
print(f"Dot product: {v1.dot(v2)}")

Vector(5, 7, 9)
Vector(3, 6, 9)
Vector(3, 6, 9)
Dot product: 32


> **WHY THIS MATTERS IN DATA SCIENCE**
> Pandas DataFrames support `len(df)`, `df[col]`, `df[mask]`, `for row in df.iterrows()` because they implement dunder methods. When you build a custom `Dataset` class for a ML project, implementing `__len__`, `__getitem__`, and `__iter__` lets it work directly with `for` loops, `random.sample`, and PyTorch's `DataLoader`.

> **INTERVIEW QUESTION**
> *"What is the difference between `__repr__` and `__str__`? Which one is called by `print()`?"*
>
> **Model answer:** `__repr__` should return an unambiguous representation — ideally one that could recreate the object. It is the fallback when `__str__` is not defined. `__str__` should return a human-readable string. `print()` calls `str()` which invokes `__str__`; if `__str__` is not defined, it falls back to `__repr__`. In f-strings, `{obj}` uses `__str__`; `{obj!r}` uses `__repr__`. The Python convention: `__repr__` for developers, `__str__` for end users.

---
## 13. Python Memory Model

Understanding how Python manages memory helps you write efficient code, avoid memory leaks, and answer system-design questions.

In [53]:
import sys
import gc

# --- Reference counting ---
# Every object has a reference count. When it hits 0, memory is reclaimed.

x = [1, 2, 3]           # ref count = 1
y = x                   # ref count = 2
z = [x, x]              # ref count = 4  (two refs from list + x + y)

print(f"Refs to x's object: {sys.getrefcount(x) - 1}")  # -1 for getrefcount's own ref

del y                   # ref count drops by 1
print(f"After del y: {sys.getrefcount(x) - 1}")

Refs to x's object: 4
After del y: 3


In [54]:
# --- Object sizes ---
print(f"int(0)   : {sys.getsizeof(0)} bytes")
print(f"int(1000): {sys.getsizeof(1000)} bytes")
print(f"float    : {sys.getsizeof(1.0)} bytes")
print(f"str 'a'  : {sys.getsizeof('a')} bytes")
print(f"str 'abc': {sys.getsizeof('abc')} bytes")
print(f"list []  : {sys.getsizeof([])} bytes")
print(f"list [1] : {sys.getsizeof([1])} bytes")
print(f"dict {{}} : {sys.getsizeof({})} bytes")

# getsizeof does NOT count nested objects (shallow)
nested = [[1,2,3], [4,5,6], [7,8,9]]
print(f"\nnested list shallow: {sys.getsizeof(nested)} bytes")
total = sys.getsizeof(nested) + sum(sys.getsizeof(row) + sum(sys.getsizeof(v) for v in row) for row in nested)
print(f"nested list total:   {total} bytes")

int(0)   : 28 bytes
int(1000): 28 bytes
float    : 24 bytes
str 'a'  : 42 bytes
str 'abc': 44 bytes
list []  : 56 bytes
list [1] : 64 bytes
dict {} : 64 bytes

nested list shallow: 80 bytes
nested list total:   596 bytes


In [55]:
# --- Cyclic garbage collection ---
# Reference counting alone cannot free circular references

class Node:
    def __init__(self, val):
        self.val  = val
        self.next = None

# Create a cycle
a = Node(1)
b = Node(2)
a.next = b
b.next = a   # cycle: a -> b -> a

# del removes our references; objects are NOT freed by refcount
# CPython's cyclic GC (generational) will eventually free them
del a, b

collected = gc.collect()   # trigger GC manually
print(f"Collected {collected} unreachable objects")

Collected 5 unreachable objects


In [56]:
# --- Memory-efficient data structures ---
# __slots__ prevents per-instance __dict__, saving memory for many small objects

class PointWithDict:     # default: has __dict__
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointWithSlots:
    __slots__ = ('x', 'y')   # no __dict__; fixed attribute set
    def __init__(self, x, y):
        self.x = x
        self.y = y

p1 = PointWithDict(1.0, 2.0)
p2 = PointWithSlots(1.0, 2.0)

print(f"With __dict__:  {sys.getsizeof(p1) + sys.getsizeof(p1.__dict__)} bytes")
print(f"With __slots__: {sys.getsizeof(p2)} bytes")
# Significant saving when you have millions of these objects

With __dict__:  344 bytes

With __slots__: 48 bytes


In [57]:
# --- Memory profiling concept (requires memory_profiler in real use) ---
# pip install memory_profiler
# from memory_profiler import profile
# @profile
# def my_function():
#     ...

# Alternative: tracemalloc (built-in)
import tracemalloc

tracemalloc.start()

# The code being measured
big_list = [i * 2 for i in range(100_000)]

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')
print("Top memory users:")
for stat in top_stats[:3]:
    print(f"  {stat}")

tracemalloc.stop()

Top memory users:
  /tmp/ipykernel_163913/2040716066.py:14: size=3903 KiB, count=99872, average=40 B
  /usr/lib/python3.12/codeop.py:126: size=478 B, count=5, average=96 B
  /home/participant/.local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3688: size=296 B, count=1, average=296 B


> **WHY THIS MATTERS IN DATA SCIENCE**
> Training on large datasets requires keeping memory usage predictable. Knowing that `del df` doesn't free memory until all references drop helps debug memory leaks in notebooks. `__slots__` matters when storing millions of lightweight feature objects. Understanding the GC helps in long-running training jobs.

> **INTERVIEW QUESTION**
> *"How does Python manage memory? What is the difference between reference counting and garbage collection?"*
>
> **Model answer:** CPython uses two mechanisms. Reference counting: every object stores a count of references pointing to it; when the count reaches zero, memory is freed immediately. This handles most objects. However, reference counting cannot free **cyclic references** (e.g., a -> b -> a, all refcounts > 0 even though unreachable). For this, CPython has a generational cyclic GC that runs periodically to detect and collect unreachable cycles. Generational means short-lived objects (gen 0) are checked most frequently, long-lived objects (gen 2) less so.

---
## 14. Underscore Conventions

Python uses underscores as a convention system for communicating intent — there is no true `private` keyword.

In [58]:
# --- Single leading underscore: _name ---
# Convention: "internal use", "not part of public API"
# Not enforced — it's a social contract

class Model:
    def __init__(self):
        self.n_features = 10        # public
        self._weights   = None      # internal — set by fit()
        self._is_fitted = False

    def fit(self, X):
        self._weights   = [0.0] * self.n_features
        self._is_fitted = True
        return self

    def _validate(self, X):     # "private" helper
        if not self._is_fitted:
            raise RuntimeError("Call fit() first")

m = Model()
print(m._weights)   # accessible but conventionally private

None


In [59]:
# --- Double leading underscore: __name — name mangling ---
# Python rewrites __name -> _ClassName__name to avoid accidental override in subclasses

class Base:
    def __init__(self):
        self.__secret = 'base_secret'   # stored as _Base__secret

    def get_secret(self):
        return self.__secret

class Derived(Base):
    def __init__(self):
        super().__init__()
        self.__secret = 'derived_secret'  # stored as _Derived__secret

d = Derived()
print(d.get_secret())            # 'base_secret' — Base's __secret is unaffected
print(d._Derived__secret)        # 'derived_secret' — name-mangled access
print(d._Base__secret)           # 'base_secret'

base_secret
derived_secret
base_secret


In [60]:
# --- Double underscore on both sides: dunder methods ---
# These are Python's protocol hooks — never invent your own __name__ attributes

class MyList:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __repr__(self):
        return f"MyList({self.data})"

ml = MyList([1, 2, 3])
print(len(ml))   # calls __len__
print(repr(ml))  # calls __repr__

3
MyList([1, 2, 3])


In [61]:
# --- Single underscore _: throwaway variable ---
# Convention: "I don't need this value"

# Ignore loop variable
for _ in range(3):
    print("tick")

# Unpack and discard
first, *_, last = [1, 2, 3, 4, 5]
print(f"first={first}, last={last}, middle discarded")

# Interactive interpreter: _ holds last expression result
x = 2 + 3
print(f"Last result (_): not useful in scripts, only in REPL")

tick
tick
tick
first=1, last=5, middle discarded
Last result (_): not useful in scripts, only in REPL


In [62]:
# --- __name__ == '__main__' pattern ---
# When a Python file is run directly, __name__ == '__main__'
# When it's imported, __name__ == the module name

# This is critical for writing reusable modules
# In a script/module:

def train():
    print("Training...")

def evaluate():
    print("Evaluating...")

if __name__ == '__main__':
    # This block ONLY runs when the file is executed directly
    # NOT when imported as: 'import my_module'
    train()
    evaluate()

print(f"Current __name__: {__name__}")   # '__main__' in this notebook

Training...
Evaluating...
Current __name__: __main__


> **WHY THIS MATTERS IN DATA SCIENCE**
> `__name__ == '__main__'` is essential for writing scripts that can be both run directly and imported by other modules (common in ML training scripts). The `_private` convention is widely used in pandas, sklearn, and PyTorch source code — recognising it tells you what's internal API vs public.

> **INTERVIEW QUESTION**
> *"What is name mangling in Python? Why does `__attr` exist?"*
>
> **Model answer:** Name mangling transforms `__attr` inside a class body to `_ClassName__attr` at compile time. This is not true privacy (the attribute is still accessible via the mangled name), but it **prevents accidental name collision in subclasses**. If `Base` uses `__config` and `Derived` also uses `__config`, they get separate namespaced attributes (`_Base__config` and `_Derived__config`). It solves a real multiple inheritance problem, not a security one.

---
## 15. Common Interview Traps

These are the gotchas that trip up experienced developers in interviews. Knowing them by name AND understanding the underlying mechanism is what impresses.

In [63]:
# ================================================================
# TRAP 1: Mutable default arguments
# ================================================================
print("=== TRAP 1: Mutable Default Arguments ===")

def append_to(element, lst=[]):
    lst.append(element)
    return lst

print(append_to(1))   # [1]
print(append_to(2))   # [1, 2]  -- NOT [2]!
print(append_to(3))   # [1, 2, 3]

# WHY: The list object [] is created once when the def is executed.
# Every call that uses the default shares this same object.
print(f"Default object id: {id(append_to.__defaults__[0])}")

=== TRAP 1: Mutable Default Arguments ===
[1]
[1, 2]
[1, 2, 3]
Default object id: 274444816293888


In [64]:
# ================================================================
# TRAP 2: Late binding closures
# ================================================================
print("=== TRAP 2: Late Binding Closures ===")

# Classic gotcha: lambda in a loop
functions_BAD = [lambda x: x + i for i in range(5)]
print([f(0) for f in functions_BAD])   # [4,4,4,4,4] — all use i=4 (final value)

# WHY: Closures capture variables by REFERENCE, not by value.
# By the time you call the functions, the loop is done and i == 4.

# Fix 1: default argument (evaluated at definition time)
functions_GOOD = [lambda x, i=i: x + i for i in range(5)]
print([f(0) for f in functions_GOOD])  # [0,1,2,3,4]

# Fix 2: factory function with its own scope
def make_adder(n):
    return lambda x: x + n

functions_FACTORY = [make_adder(i) for i in range(5)]
print([f(0) for f in functions_FACTORY])  # [0,1,2,3,4]

=== TRAP 2: Late Binding Closures ===
[4, 4, 4, 4, 4]
[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4]


In [65]:
# ================================================================
# TRAP 3: Chained comparisons (actually a feature, not a bug)
# ================================================================
print("=== TRAP 3: Chained Comparisons ===")

# Python supports chained comparisons — often surprising to other-language devs
x = 5
print(1 < x < 10)      # True  -- Pythonic range check!
print(1 < x < 4)       # False
print(1 < 3 > 2)       # True  -- both (1<3) AND (3>2) must hold

# The TRAP: forgetting this isn't how C/Java works
# In C: (1 < x < 10) would be ((1 < x) < 10) = (True < 10) = (1 < 10) = True always!
# Python evaluates it as: (1 < x) AND (x < 10) with x evaluated only once

# Tricky examples:
print(False == False == False)  # True: (False==False) AND (False==False)
print(True == True == False)    # False: True but (True==False) is False

=== TRAP 3: Chained Comparisons ===
True
False
True
True
False


In [66]:
# ================================================================
# TRAP 4: Integer caching
# ================================================================
print("=== TRAP 4: Integer Caching ===")

# CPython caches integers from -5 to 256
a = 100; b = 100
print(f"100 is 100: {a is b}")   # True — same cached object

c = 1000; d = 1000
print(f"1000 is 1000: {c is d}")  # False — different objects (outside cache)

# The lesson: NEVER use 'is' to compare values. 'is' is for identity.
# ALWAYS use == for value comparison.
print(f"1000 == 1000: {c == d}")  # True — always correct for values

=== TRAP 4: Integer Caching ===
100 is 100: True
1000 is 1000: False
1000 == 1000: True


In [67]:
# ================================================================
# TRAP 5: List multiplication with mutable objects
# ================================================================
print("=== TRAP 5: List Multiplication ===")

# Looks like creating a 2D grid
grid_BAD = [[0] * 3] * 3   # creates 3 references to THE SAME inner list
grid_BAD[0][0] = 99
print("BAD grid:", grid_BAD)   # [[99,0,0],[99,0,0],[99,0,0]] — all rows changed!

# Correct: use list comprehension to create independent rows
grid_GOOD = [[0] * 3 for _ in range(3)]
grid_GOOD[0][0] = 99
print("GOOD grid:", grid_GOOD)  # [[99,0,0],[0,0,0],[0,0,0]]

=== TRAP 5: List Multiplication ===
BAD grid: [[99, 0, 0], [99, 0, 0], [99, 0, 0]]
GOOD grid: [[99, 0, 0], [0, 0, 0], [0, 0, 0]]


In [68]:
# ================================================================
# TRAP 6: for-else and while-else
# ================================================================
print("=== TRAP 6: for-else ===")

# The else clause on a loop runs if the loop completed WITHOUT a break
def find_outlier(data, threshold=3.0):
    for i, val in enumerate(data):
        if abs(val) > threshold:
            print(f"Outlier found at index {i}: {val}")
            break
    else:
        # Only reaches here if no break occurred
        print("No outliers found")

find_outlier([0.5, 1.2, 0.8, 2.9])
find_outlier([0.5, 1.2, 0.8, 5.0])

=== TRAP 6: for-else ===
No outliers found
Outlier found at index 3: 5.0


In [69]:
# ================================================================
# TRAP 7: is vs == for None
# ================================================================
print("=== TRAP 7: None comparison ===")

# None is a singleton — there is exactly one None object
# Correct: use 'is' for None checks (PEP 8 mandates this)
val = None
print(f"val is None: {val is None}")   # Correct
print(f"val == None: {val == None}")   # Works but wrong idiom

# WHY 'is' is correct: custom classes can override __eq__
# and make 'obj == None' return True (e.g., numpy arrays)
import numpy as np
arr = np.array([1, 2, 3])
# arr == None  would return array([False, False, False]), not a single bool
print(f"arr is None: {arr is None}")  # False — correct and unambiguous

=== TRAP 7: None comparison ===
val is None: True
val == None: True
arr is None: False


In [70]:
# ================================================================
# TRAP 8: Modifying a list while iterating over it
# ================================================================
print("=== TRAP 8: Modifying while iterating ===")

data_BAD = [1, 2, 3, 4, 5, 6]
for item in data_BAD:
    if item % 2 == 0:
        data_BAD.remove(item)   # skips elements!
print(f"BAD result: {data_BAD}")   # [1, 3, 5] — may look right but is accidental

data_GOOD = [1, 2, 3, 4, 5, 6]
result = [item for item in data_GOOD if item % 2 != 0]  # create new list
print(f"GOOD result: {result}")

=== TRAP 8: Modifying while iterating ===
BAD result: [1, 3, 5]
GOOD result: [1, 3, 5]


> **WHY THIS MATTERS IN DATA SCIENCE**
> These traps appear in data pipelines constantly. Late-binding closures bite when dynamically building transformation functions in feature engineering. Mutable default args corrupt caches. Grid initialization traps create subtle bugs in DP algorithms. The `is None` trap matters because numpy arrays override `==`.

> **INTERVIEW QUESTION**
> *"What will `[lambda x: x + i for i in range(3)]` produce when each function is called with argument 0? Why? How do you fix it?"*
>
> **Model answer:** All three functions return `2` (the final value of `i`). Closures close over the **variable** `i`, not its value at the time the lambda was created. By the time you call the functions, the for loop has completed and `i == 2`. Fix: use a default argument `lambda x, i=i: x + i` — default argument values are evaluated at the time the lambda is defined, capturing the current value. Alternatively, use a factory function whose parameter creates a new scope binding.

---
## Final Review: Quick-Reference Cheat Sheet

| Topic | Key Point | Common Trap |
|-------|-----------|-------------|
| List vs Generator | Generator is lazy, O(1) memory | Generator exhausts after one pass |
| Mutable/Immutable | Lists/dicts mutate in place | `[]` default arg is shared across calls |
| `*args`/`**kwargs` | tuple / dict; order matters | Forgetting `@functools.wraps` in decorators |
| Decorators | `func = decorator(func)` | `@property` doesn't need `()` at call site |
| Generators | `yield` suspends frame | Can't restart a generator; `send()` to communicate |
| Context Managers | `__exit__` returning `True` suppresses exceptions | `finally` always runs, `else` only on success |
| Lambda/map/filter | `map` is lazy in Python 3 | Prefer comprehensions for readability |
| zip/enumerate | `zip` stops at shortest; returns iterator | Iterator exhausts — can't reuse without `list()` |
| dict tricks | `Counter`, `defaultdict`, `\|` merge | `.get()` vs `[]` — use `.get()` for safe access |
| f-strings | Fastest formatting; `=` specifier for debug | `!r` applies `repr()` |
| Exceptions | `else` = no exception; `finally` = always | `raise X from Y` preserves causal chain |
| OOP dunders | `__repr__` for devs, `__str__` for users | `__eq__` should return `NotImplemented`, not `False` |
| Memory | refcount + cyclic GC; `__slots__` saves memory | `del` only drops reference; `sys.getsizeof` is shallow |
| Underscores | `_` = internal; `__` = name mangled; `__x__` = dunder | Name mangling is not security |
| Traps | Late binding, mutable defaults, `is` vs `==` | Never `is` for values; `is None` is correct |

In [71]:
# ============================================================
# BONUS: Quick self-test
# Run this cell to verify you can answer these from memory.
# ============================================================

questions = [
    "1.  What is the memory difference between [x for x in range(1M)] and (x for x in range(1M))?",
    "2.  Why is def f(lst=[]) dangerous? What is the fix?",
    "3.  What is the required order for def f(a, *args, kw, **kwargs)?",
    "4.  What does @functools.wraps do and why is it important?",
    "5.  What happens to generator state when execution is between yield calls?",
    "6.  What does __exit__ returning True do?",
    "7.  When would you use map() over a list comprehension?",
    "8.  What does zip(*list_of_tuples) accomplish?",
    "9.  What is the difference between d.get('k') and d['k']?",
    "10. What does f'{value=}' print? (Python 3.8+)",
    "11. When does the 'else' clause of a try block run?",
    "12. What is name mangling and why does Python have it?",
    "13. How does CPython free circular references?",
    "14. Why should you use 'is' not '==' to check for None?",
    "15. What will [lambda x: x+i for i in range(3)][0](0) return?",
]

print("=== SELF-TEST: Advanced Python Interview Questions ===")
for q in questions:
    print(q)

=== SELF-TEST: Advanced Python Interview Questions ===
1.  What is the memory difference between [x for x in range(1M)] and (x for x in range(1M))?
2.  Why is def f(lst=[]) dangerous? What is the fix?
3.  What is the required order for def f(a, *args, kw, **kwargs)?
4.  What does @functools.wraps do and why is it important?
5.  What happens to generator state when execution is between yield calls?
6.  What does __exit__ returning True do?
7.  When would you use map() over a list comprehension?
8.  What does zip(*list_of_tuples) accomplish?
9.  What is the difference between d.get('k') and d['k']?
10. What does f'{value=}' print? (Python 3.8+)
11. When does the 'else' clause of a try block run?
12. What is name mangling and why does Python have it?
13. How does CPython free circular references?
14. Why should you use 'is' not '==' to check for None?
15. What will [lambda x: x+i for i in range(3)][0](0) return?
